# LAB | Relevance Scoring and Rerankers

This notebook builds a basic RAG retrieval pipeline, then compares retrieval quality before and after metadata filtering and reranking.

## Step 1: Data Loading

Load the Trustworthy AI transcript and the PDF document, then prepare them for chunking and retrieval.

In [1]:
from pathlib import Path
from collections import Counter
import os

import numpy as np
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from openai import OpenAI

DATA_DIR = Path("data")

print(list(DATA_DIR.iterdir()))

[WindowsPath('data/hleg_ethics_guidelines_for_trustworthy_ai-en.pdf'), WindowsPath('data/trustworthy_ai_transcript.txt')]


In [2]:
transcript_path = DATA_DIR / "trustworthy_ai_transcript.txt"

with open(transcript_path, "r", encoding="utf-8") as f:
    transcript_text = f.read()

print(f"Transcript length: {len(transcript_text):,} characters")
print(transcript_text[:500])

Transcript length: 18,622 characters
0:00: So imagine for a second you're driving across a, I don't know, a massive suspension bridge. 
 0:06: You don't pull over halfway across, get out, and demand to see the blueprints, right? 
 0:10: You don't, , interview the welding crew. 
 0:12: No, you just, you trust it. 
 0:14: You just drive. 
 0:15: You trust the bridge. 
 0:15: You trust the engineering standards, the inspection. 
 0:18: The laws that say this thing won't fail, right? 
 0:20: It's trust in the infrastructure. 
 0:22: It


In [3]:
pdf_path = DATA_DIR / "hleg_ethics_guidelines_for_trustworthy_ai-en.pdf"

reader = PdfReader(pdf_path)

pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print(f"PDF length: {len(pdf_text):,} characters")
print(pdf_text[:500])

PDF length: 158,079 characters
 
  
 
 
 
INDEPENDENT  
HIGH-LEVEL EXPERT GROUP ON  
ARTIFICIAL INTELLIGENCE  
SET UP BY THE EUROPEAN COMMISSION  
 
 
 
 
 
 
ETHICS GUIDELINES  
FOR TRUSTWORTHY AI 
 
 
 
  
 
ETHICS GUIDELINES  FOR  TRUSTWORTHY  AI 
 
High -Level Expert Group on Artificial Intelligence  
 
 
 
 
 
 
 
 
This document was written by the High -Level Expert Group on AI (AI HLEG) . The members of the AI HLEG 
named in this document support the overall framework for Trustworthy  AI put forward in these Guidelines


In [4]:
documents = [
    {
        "text": transcript_text,
        "source": "transcript"
    },
    {
        "text": pdf_text,
        "source": "pdf"
    }
]

for doc in documents:
    print(doc["source"], len(doc["text"]))

transcript 18622
pdf 158079


In [5]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for doc in documents:
    text = doc["text"]

    for i in range(0, len(text), CHUNK_SIZE - CHUNK_OVERLAP):
        chunk_text = text[i:i + CHUNK_SIZE]

        chunks.append({
            "text": chunk_text,
            "source": doc["source"]
        })

print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:3]:
    print(chunk["source"], len(chunk["text"]))

Total chunks: 222
transcript 1000
transcript 1000
transcript 1000


In [6]:
from collections import Counter

source_counts = Counter(chunk["source"] for chunk in chunks)

print(source_counts)

Counter({'pdf': 198, 'transcript': 24})


In [7]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [8]:
client = OpenAI(api_key=api_key)

print("OpenAI client ready")

OpenAI client ready


In [9]:
test_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks[0]["text"]
)

print(len(test_embedding.data[0].embedding))

1536


In [10]:
embeddings = []

for i, chunk in enumerate(chunks):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk["text"]
    )

    embeddings.append(response.data[0].embedding)

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(chunks)}")

print(f"Finished: {len(embeddings)} embeddings")

Processed 25/222
Processed 50/222
Processed 75/222
Processed 100/222
Processed 125/222
Processed 150/222
Processed 175/222
Processed 200/222
Finished: 222 embeddings
